# Task: GNN training on synthetic SPV simulations: adjacency, cell state and property matrices

We will be developing a graph neural network (GNN)-based model capable of inferring mechanistic rules and uncovering the principles driving DPAC aggregation. To facilitate this, the GNN will initially be trained using synthetic Self-Propelled Voronoi (SPV) simulations, serving as placeholder data while the deep learning infrastructure is optimized. The GNN will be validated by its ability to, first, recover the physical mechanisms embedded in the SPV model, then subsequently applied to DPAC data to explore the impacts of initial thickness and cell density.

### GNN training

In [13]:
import os
import numpy as np
import importlib.util
import pickle
from scipy.sparse import coo_matrix
from spektral.data import Graph


def load_params(py_path):
    """
    Dynamically loads a Python file containing parameters, e.g.:
        v0 = [0.1, 1.3]
        W = [[0.0, 0.08],
             [0.08, 0.0]]
        A0 = [0.9, 0.9]
        P0 = [3.812, 3.812]
        Dr = 50
        kappa_A = 0.4
        kappa_P = 0.07
        a = 0.25
        k = 2.5
    """
    print(f"> Attempting to load parameters from: {py_path}")
    spec = importlib.util.spec_from_file_location("param_module", py_path)
    param_module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(param_module)
    
    # Gather the parameters into a dictionary:
    p = {
        "v0": getattr(param_module, "v0", None),
        "W": getattr(param_module, "W", None),
        "A0": getattr(param_module, "A0", None),
        "P0": getattr(param_module, "P0", None),
        "Dr": getattr(param_module, "Dr", None),
        "kappa_A": getattr(param_module, "kappa_A", None),
        "kappa_P": getattr(param_module, "kappa_P", None),
        "a": getattr(param_module, "a", None),
        "k": getattr(param_module, "k", None),
    }
    print(f"> Loaded parameters: {p}")
    return p


def build_dataset(
    data_dir: str,
    param_path: str,
    t_start: int = 0,
    t_end: int = 200,
    t_step: int = 1,
):
    """
    Build a dataset of Graph snapshots (G_t) from time t in [t_start, t_end],
    stepping by t_step. For each t, we also produce the adjacency at t+1 (if 
    available) for training a GNN that predicts adjacency changes from t -> t+1.

    Key difference: we only gather "unique" edges (i<j), then duplicate them 
    to produce 2E edges in a well-defined forward-block + reverse-block.

    Returns:
      A list of (Graph, adjacency_{t+1}) for times t.
    """
    print(f"\n=== build_dataset ===")
    print(f"Data dir  = {data_dir}")
    print(f"Param file= {param_path}")
    print(f"Time range= [{t_start}, {t_end}] with step={t_step}\n")

    # 1) Load the param file
    p = load_params(param_path)
    
    # 2) We'll store a list of (G_t, adjacency_{t+1}) for each valid t
    dataset = []

    # 3) Iterate over time steps
    t_list = range(t_start, t_end + 1, t_step)
    print(f">> Will look for data_{{t}}.npy from t in {list(t_list)}")

    for t in t_list:
        data_file_t = os.path.join(data_dir, f"data_{t}.npy")
        if not os.path.exists(data_file_t):
            print(f" - data_{t}.npy not found. Skipping.")
            continue

        print(f"\nLoading timepoint t={t} from {data_file_t}")
        data_t = np.load(data_file_t, allow_pickle=True).item()

        # We'll only build a label adjacency if (t + t_step) also exists
        t_next = t + t_step
        data_file_tplus = os.path.join(data_dir, f"data_{t_next}.npy")
        if os.path.exists(data_file_tplus):
            data_tplus = np.load(data_file_tplus, allow_pickle=True).item()
            adj_label = data_tplus["cell_adj"]  # shape [n_c, n_c]
            print(f"   Found adjacency label at t+{t_step} = {t_next}")
        else:
            adj_label = None
            print(f"   No data for t+{t_step} = {t_next}. Label is None.")

        # Extract your raw data from dictionary:
        cell_x    = data_t["cell_x"]           # shape [n_c, 2]
        cell_type = data_t["cell_type"]        # shape [n_c]
        area      = data_t["area"]             # shape [n_c]
        perimeter = data_t["perimeter"]        # shape [n_c]
        cell_adj  = data_t["cell_adj"]         # shape [n_c, n_c]

        # "edge_voronoi_length" might or might not exist
        voronoi_len = data_t.get("edge_voronoi_length", None)
        if voronoi_len is None:
            print("   edge_voronoi_length not found in data dict.")
        else:
            print("   Found edge_voronoi_length in data dict.")

        n_c = cell_x.shape[0]
        print(f"   n_c = {n_c} cells in this timepoint")

        # 4) Build node features for time t
        #    Example: [ x_i, y_i, area_i, perimeter_i, cell_type_i, 
        #               A0(type), P0(type), Dr, kappa_A, kappa_P, a, k ]
        node_feats = []
        for i in range(n_c):
            ctype = cell_type[i]
            A0_val = p["A0"][ctype]
            P0_val = p["P0"][ctype]
            (x_i, y_i) = cell_x[i]
            feats = [
                x_i,
                y_i,
                area[i],
                perimeter[i],
                float(ctype),
                A0_val,
                P0_val,
                p["Dr"],
                p["kappa_A"],
                p["kappa_P"],
                p["a"],
                p["k"],
            ]
            node_feats.append(feats)
        node_feats = np.array(node_feats, dtype=np.float32)  # shape [n_c, D]

        # 5) Build adjacency + edge features, but first we gather only "unique" edges (i<j)
        unique_rows = []
        unique_cols = []
        unique_edge_feats = []

        for i in range(n_c):
            for j in range(i+1, n_c):
                if cell_adj[i, j] == 1:
                    # i < j, this is the unique edge
                    unique_rows.append(i)
                    unique_cols.append(j)
                    
                    # Voronoi length if available
                    vlen = 0.0
                    if voronoi_len is not None:
                        vlen = voronoi_len[i, j]
                    
                    # W_interaction for this pair
                    ctype_i = cell_type[i]
                    ctype_j = cell_type[j]
                    Wij = p["W"][ctype_i][ctype_j]
                    
                    # Build the edge feature for i->j
                    unique_edge_feats.append([vlen, Wij])

        E_unique = len(unique_rows)  # number of unique edges

        # 6) Duplicate them to get forward + reverse
        rows_duplicated = unique_rows + unique_cols  # length 2*E_unique
        cols_duplicated = unique_cols + unique_rows  # same length
        edge_feats_duplicated = unique_edge_feats + unique_edge_feats  # also length 2*E_unique

        row_ar = np.array(rows_duplicated, dtype=np.int32)
        col_ar = np.array(cols_duplicated, dtype=np.int32)
        edge_feats_ar = np.array(edge_feats_duplicated, dtype=np.float32)

        data_ar = np.ones_like(row_ar, dtype=np.float32)  # or whatever weight you want
        a_coo = coo_matrix((data_ar, (row_ar, col_ar)), shape=(n_c, n_c))

        # 7) Construct the Spektral Graph for time t
        #    shape of edge_feats_ar => [2*E_unique, D_edge]
        G_t = Graph(
            x=node_feats,
            a=a_coo,
            e=edge_feats_ar
        )
        
        # 8) Append (G_t, adj_label) to dataset
        dataset.append((G_t, adj_label))

    print(f"\nBuild complete. Total {len(dataset)} snapshots processed.")
    return dataset

In [14]:

if __name__ == "__main__":
    # Example usage with your lists of data_dirs and param_files.
    data_dirs = [
        f"/Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/output_Fig5I/matrix_output_{i}"
        for i in range(1, 11)
    ]
    param_files = [
        f"/Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/DPAC1_parameters/{i}.py"
        for i in range(1, 11)
    ]

    # We'll define some time config:
    T_START = 0
    T_END = 2000   # up to 2000
    T_STEP = 200     # maybe your data are spaced in increments of 4

    # We'll build dataset for each sim ID in turn
    all_datasets = []
    for sim_idx in range(10):
        ddir = data_dirs[sim_idx]
        pfile = param_files[sim_idx]
        ds = build_dataset(
            data_dir=ddir,
            param_path=pfile,
            t_start=T_START,
            t_end=T_END,
            t_step=T_STEP
        )
        print(
            f"Simulation ID = {sim_idx+1}: built dataset with {len(ds)} steps "
            f"from directory '{ddir}'."
        )
        all_datasets.append(ds)

    # Now save `all_datasets` to a local file (e.g., "all_datasets.pkl"):

    save_path = "all_datasets.pkl"
    print(f"\nSaving all_datasets to {save_path} ...")
    with open(save_path, "wb") as f:
        pickle.dump(all_datasets, f)

    print("Done. The file contains a list of 10 datasets (one per simulation).")

    # Now all_datasets is a list of datasets, each one being a list of (Graph, adj_label).
    # all_datasets[i] corresponds to simulation i+1.
    
    # Example: just checking the first step from simulation #1
    if len(all_datasets[0]) > 0:
        graph_0, adj_label_0 = all_datasets[0][0]
        print("First Graph node features shape:", graph_0.x.shape)
        print("First Graph adjacency shape:", graph_0.a.shape)
        print("First Graph edge feature shape:", graph_0.e.shape)
        if adj_label_0 is not None:
            print("Next adjacency shape:", adj_label_0.shape)



=== build_dataset ===
Data dir  = /Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/output_Fig5I/matrix_output_1
Param file= /Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/DPAC1_parameters/1.py
Time range= [0, 2000] with step=200

> Attempting to load parameters from: /Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/DPAC1_parameters/1.py
> Loaded parameters: {'v0': [0.1, 1.3], 'W': ([0.0, 0.08], [0.08, 0.0]), 'A0': [0.9, 0.9], 'P0': [3.812, 3.812], 'Dr': 50, 'kappa_A': 0.4, 'kappa_P': 0.07, 'a': 0.25, 'k': 2.5}
>> Will look for data_{t}.npy from t in [0, 200, 400, 600, 800, 1000, 1200, 1400, 1600, 1800, 2000]

Loading timepoint t=0 from /Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/output_Fig5I/matrix_output_1/data_0.npy
   Found adjacency label at t+200 = 200
   Found edge_voronoi_length in data dict.
   n_c = 840 cells in this timepoint

Loading timepoint t=200 from /Users/sophia01px2019/Downloads/gartner_lab_rotati

Here is where we:

Load all_datasets.pkl.
For each (Graph_t, adj_label_{t+1}), convert to PyG format.
Use NeighborLoader to sample subgraphs for each node (or a fraction of them).
Build next-step labels (adj, area, perimeter, boundary length, etc.) restricted to subgraph nodes.
Save subgraph+label to some file for training.

In [22]:
# Feature layouts: 

# node_feats[i] = [
#     x_i,       # index 0
#     y_i,       # index 1
#     area[i],   # index 2
#     perimeter[i], # 3
#     cell_type[i], # 4
#     A0_val,    # 5
#     P0_val,    # 6
#     Dr,        # 7
#     kappa_A,   # 8
#     kappa_P,   # 9
#     a,         # 10
#     k,         # 11
# ]
# edge_feats[e] = [
#     vlen,  # index 0 => edge_voronoi_length
#     Wij,   # index 1 => W_interaction
# ]

In [3]:
import torch_scatter
import torch_sparse
import torch_cluster
import torch_spline_conv
import torch_geometric

In [26]:
import os
import pickle
import numpy as np
import torch
from torch_geometric.data import Data

def convert_spektral_to_pyg(spektral_graph):
    """
    Convert a spektral Graph object to PyG Data.
    """
    x = torch.FloatTensor(spektral_graph.x)  # shape [N, D_node]

    # Convert adjacency from spektral (coo_matrix) => PyG edge_index
    row = spektral_graph.a.row
    col = spektral_graph.a.col
    edge_index = np.vstack([row, col])  # shape [2, E]
    edge_index_t = torch.LongTensor(edge_index)

    # Edge features
    edge_attr = torch.FloatTensor(spektral_graph.e) if spektral_graph.e is not None else None

    pyg_data = Data(
        x=x,
        edge_index=edge_index_t,
        edge_attr=edge_attr,
    )
    return pyg_data


def build_fullgraph_label(
    pyg_data_t,
    adjacency_tplus,    # shape [N, N], binary adjacency at t+1
    x_time_t,           # shape [N, D_node], node feats at time t
    x_time_tplus,       # shape [N, D_node], node feats at time t+1
    e_time_t=None,      # shape [E, D_edge], edge feats at time t (one per edge in spektral form)
    e_time_tplus=None,  # shape [E, D_edge], edge feats at time t+1 (like boundary length)
    row_t=None,
    col_t=None,
    row_tplus=None,
    col_tplus=None
):
    """
    Builds a label dict for the *entire* graph (not subgraphs).
    We assume:
      - x_time_t, x_time_tplus: per-node arrays with both "constant" features (cell type, etc.)
        and "dynamic" features (area, perimeter, coords).
      - adjacency_tplus: next-step adjacency for the entire graph
      - e_time_t, e_time_tplus: edge-level arrays (like [2E, ...] or [E, ...]) if needed
      - row_t, col_t: the row/col arrays from spektral_graph_t.a (coo) at time t
      - row_tplus, col_tplus: row/col from G_{t+1}, if you want next-step boundary length

    We'll return a dictionary with:
      node_constants, node_labels, edge_constants, edge_exists, edge_length, ...
    So your GNN can train to predict adjacency(t+1), geometry(t+1), etc.
    """

    N = pyg_data_t.num_nodes
    E = pyg_data_t.num_edges

    # 1) Node constants: e.g. cell_type, A0, P0, Dr, ...
    #    Suppose columns in x_time_t are:
    #      0-> x, 1-> y, 2-> area, 3-> perimeter, 4-> cell_type, 5-> A0, ...
    node_constants = {}
    # Example: store them as separate arrays or a single matrix
    node_constants["cell_type"] = x_time_t[:, 4]
    node_constants["A0"]        = x_time_t[:, 5]
    node_constants["P0"]        = x_time_t[:, 6]
    node_constants["Dr"]        = x_time_t[:, 7]
    node_constants["kappa_A"]   = x_time_t[:, 8]
    node_constants["kappa_P"]   = x_time_t[:, 9]
    node_constants["a"]         = x_time_t[:,10]
    node_constants["k"]         = x_time_t[:,11]

    # 2) Node labels: geometry at t+1 => coords, area, perimeter
    node_labels = {}
    # E.g. x_time_tplus col 0-> x, 1-> y, 2-> area, 3-> perimeter
    node_labels["coords"]    = x_time_tplus[:, 0:2]
    node_labels["area"]      = x_time_tplus[:, 2]
    node_labels["perimeter"] = x_time_tplus[:, 3]

    # 3) Edge-level constants (like W) from e_time_t
    #    We'll store them in an array of shape [E], each edge from pyg_data_t.edge_index
    edge_constants = None
    if e_time_t is not None and row_t is not None:
        edge_constants = np.zeros(E, dtype=np.float32)
        # row_t, col_t => define the graph at time t
        # e_time_t might be shape [E2, D_edge], or [2E2, D_edge]. 
        # We'll assume [E2, D_edge] where E2 = #edges in spektral.
        # We'll do a dictionary mapping (row_t[k], col_t[k]) => e_time_t[k, ...]
        # Then read them out in the order of pyg_data_t.edge_index
        # That is row_sub = pyg_data_t.edge_index[0,e], col_sub = ...
        
        # Build a quick map from (r,c) to e_time_t index
        # but remember the graph is undirected => we might do (min(r,c), max(r,c)).
        edge_map = {}
        E_spektral = len(row_t)
        for k in range(E_spektral):
            r = row_t[k]
            c = col_t[k]
            if r > c:
                r, c = c, r
            # Suppose the second column e_time_t[k,1] is W or so:
            edge_map[(r,c)] = e_time_t[k]

        # Now fill edge_constants
        for e_i in range(E):
            src = pyg_data_t.edge_index[0,e_i].item()
            dst = pyg_data_t.edge_index[1,e_i].item()
            # undirected => sort
            if src > dst:
                src, dst = dst, src
            if (src,dst) in edge_map:
                # let's say the W param is edge_map[(src,dst)][1]
                # or if your W is at index 0
                edge_constants[e_i] = edge_map[(src,dst)][1]  
            else:
                edge_constants[e_i] = 0.0

    # 4) Next-step adjacency => edge_exists, shape [E], 0/1
    #    If adjacency_tplus[src,dst] = 1 => edge_exists=1
    edge_exists = np.zeros(E, dtype=np.float32)
    for e_i in range(E):
        src = pyg_data_t.edge_index[0,e_i].item()
        dst = pyg_data_t.edge_index[1,e_i].item()
        if adjacency_tplus[src, dst] == 1:
            edge_exists[e_i] = 1.0

    # 5) Next-step edge length if e_time_tplus is not None
    edge_length = None
    if e_time_tplus is not None and row_tplus is not None:
        edge_length = np.zeros(E, dtype=np.float32)
        # build map from (r,c) => e_time_tplus, same logic as above
        edge_map_tp1 = {}
        E_plus = len(row_tplus)
        for k in range(E_plus):
            r = row_tplus[k]
            c = col_tplus[k]
            if r > c:
                r, c = c, r
            edge_map_tp1[(r,c)] = e_time_tplus[k]

        for e_i in range(E):
            src = pyg_data_t.edge_index[0,e_i].item()
            dst = pyg_data_t.edge_index[1,e_i].item()
            if src > dst:
                src, dst = dst, src
            if (src,dst) in edge_map_tp1:
                # Suppose boundary length is e_map_tp1[(src,dst)][0]
                edge_length[e_i] = edge_map_tp1[(src,dst)][0]
            # else => 0.0 default

    # 6) Combine into final label dict
    label_dict = {
        "node_constants": node_constants,
        "node_labels":    node_labels,   # next-step geometry
        "edge_exists":    edge_exists    # next-step adjacency
    }
    if edge_constants is not None:
        label_dict["edge_constants"] = edge_constants
    if edge_length is not None:
        label_dict["edge_length"] = edge_length

    return label_dict


def fullgraph_for_dataset(
    all_datasets_path,
    out_dir
):
    """
    1) Load 'all_datasets' from all_datasets_path (a list of simulation runs).
       Each simulation is a list of (spektral_graph_t, adj_label_tplus).
    2) For each time t, convert the entire G_t => PyG (no subgraph partition).
    3) Build a single label dict for the entire graph:
       - next-step adjacency => edge_exists,
       - next-step geometry => node_labels (area, perimeter, coords),
       - constants => cell_type, W param, etc.
    4) Save (pyg_data_t, label_dict_tplus) to out_dir for training or inference.
    """

    import torch
    os.makedirs(out_dir, exist_ok=True)

    # ---------------- LOAD ----------------
    with open(all_datasets_path, "rb") as f:
        all_datasets = pickle.load(f)

    print(f"Loaded all_datasets from '{all_datasets_path}'. Found {len(all_datasets)} sims.\n")

    for sim_idx, ds in enumerate(all_datasets):
        # ds = [ (G_t0, adj0plus), (G_t1, adj1plus), ... ]
        n_timepoints = len(ds)
        print(f"=== Simulation #{sim_idx+1}: {n_timepoints} timepoints. ===")

        for t_idx in range(n_timepoints - 1):
            (spektral_graph_t, adj_label_tplus) = ds[t_idx]
            if adj_label_tplus is None:
                # no adjacency for t+1 => skip
                continue

            # We want to also get G_{t+1} for node feats if needed
            (spektral_graph_tplus, _) = ds[t_idx+1]

            # Convert entire G_t => PyG
            pyg_data_t = convert_spektral_to_pyg(spektral_graph_t)

            # We also want to gather arrays for build_fullgraph_label
            adjacency_tplus = adj_label_tplus               # shape [N, N]
            x_time_t       = spektral_graph_t.x             # shape [N, D_node]
            e_time_t       = spektral_graph_t.e
            row_t          = getattr(spektral_graph_t.a, "row", None)
            col_t          = getattr(spektral_graph_t.a, "col", None)

            x_time_tplus   = spektral_graph_tplus.x
            e_time_tplus   = spektral_graph_tplus.e
            row_tplus      = getattr(spektral_graph_tplus.a, "row", None)
            col_tplus      = getattr(spektral_graph_tplus.a, "col", None)

            label_dict = build_fullgraph_label(
                pyg_data_t,
                adjacency_tplus,
                x_time_t, x_time_tplus,
                e_time_t, e_time_tplus,
                row_t, col_t,
                row_tplus, col_tplus
            )

            # Now we have (pyg_data_t, label_dict) for the entire time t
            # Save it
            out_name = f"fullgraph_sim{sim_idx+1}_t{t_idx}.pkl"
            out_path = os.path.join(out_dir, out_name)
            with open(out_path, "wb") as f_out:
                pickle.dump((pyg_data_t, label_dict), f_out)

            print(f"   Saved [Sim={sim_idx+1}, t={t_idx}] => {out_path}")

        print("-----------------------------------------------------")

    print("Done building full-graph dataset. Each time step => one (pyg_data, label_dict).")


In [28]:

# if __name__ == "__main__":

#     ALL_DATASETS_PATH = "all_datasets.pkl"  
#     # file you previously created that holds the list of datasets: 
#     #   all_datasets[sim_index] => list of (Graph, adjacency_label)

#     OUT_SUBGRAPHS_DIR = "subgraphs_output"  
#     # We'll store the neighbor-sampled subgraphs in this directory.

#     # If you want to specify neighbor-sampler parameters:
#     NUM_NEIGHBORS = [6, 6, 6]  # e.g. 3-layer sampling: 6 neighbors for the first node, then 6 neighbors per each of these 6 1st layer neighbors. If neighbor # < # defined then takes all neighbors. 
#     BATCH_SIZE = 1           # If 1, each subgraph is "centered" on a single node per iteration

#     neighbor_sampling_for_dataset(
#         all_datasets_path=ALL_DATASETS_PATH,   # path to 'all_datasets.pkl'
#         out_dir=OUT_SUBGRAPHS_DIR,
#         num_neighbors=NUM_NEIGHBORS, # length can be flexibly adjusted to increase or decrease layers
#         batch_size=BATCH_SIZE
#     )

#     print("Neighbor sampling complete.")

if __name__ == "__main__":
    data_dir = "all_datasets.pkl"  
    out_dir = "fullgraphs_output" 
    fullgraph_for_dataset(data_dir, out_dir)


Loaded all_datasets from 'all_datasets.pkl'. Found 10 sims.

=== Simulation #1: 11 timepoints. ===
   Saved [Sim=1, t=0] => fullgraphs_output/fullgraph_sim1_t0.pkl
   Saved [Sim=1, t=1] => fullgraphs_output/fullgraph_sim1_t1.pkl
   Saved [Sim=1, t=2] => fullgraphs_output/fullgraph_sim1_t2.pkl
   Saved [Sim=1, t=3] => fullgraphs_output/fullgraph_sim1_t3.pkl
   Saved [Sim=1, t=4] => fullgraphs_output/fullgraph_sim1_t4.pkl
   Saved [Sim=1, t=5] => fullgraphs_output/fullgraph_sim1_t5.pkl
   Saved [Sim=1, t=6] => fullgraphs_output/fullgraph_sim1_t6.pkl
   Saved [Sim=1, t=7] => fullgraphs_output/fullgraph_sim1_t7.pkl
   Saved [Sim=1, t=8] => fullgraphs_output/fullgraph_sim1_t8.pkl
   Saved [Sim=1, t=9] => fullgraphs_output/fullgraph_sim1_t9.pkl
-----------------------------------------------------
=== Simulation #2: 11 timepoints. ===
   Saved [Sim=2, t=0] => fullgraphs_output/fullgraph_sim2_t0.pkl
   Saved [Sim=2, t=1] => fullgraphs_output/fullgraph_sim2_t1.pkl
   Saved [Sim=2, t=2] => full

In [30]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class EdgeFeatureConv(nn.Module):
    """
    A single message-passing layer that uses node embeddings + edge attributes.
    message_ij = MLP(h_i || h_j || e_ij)
    aggregated_j = sum_{i in N(j)} message_ij
    h_j_new = ReLU(aggregated_j)
    """

    def __init__(self, node_dim, edge_dim, hidden_dim):
        """
        :param node_dim: dimension of node embeddings
        :param edge_dim: dimension of edge features
        :param hidden_dim: dimension of output embeddings
        """
        super().__init__()
        in_dim = node_dim * 2 + edge_dim
        self.message_mlp = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

    def forward(self, x, edge_index, edge_attr):
        """
        :param x: shape [N, node_dim]  (node embeddings)
        :param edge_index: shape [2, E]
        :param edge_attr: shape [E, edge_dim]
        :return: x_out: [N, hidden_dim]
        """
        src = edge_index[0]  # [E]
        dst = edge_index[1]  # [E]

        h_src = x[src]  # [E, node_dim]
        h_dst = x[dst]  # [E, node_dim]

        # cat => shape [E, node_dim*2 + edge_dim]
        msg_input = torch.cat([h_src, h_dst, edge_attr], dim=1)
        messages = self.message_mlp(msg_input)  # [E, hidden_dim]

        # sum over destination
        N = x.shape[0]
        out = torch.zeros(N, messages.shape[1], device=x.device)
        out.index_add_(0, dst, messages)  # sum into out[dst]

        return F.relu(out)


In [31]:
class GNCAFullGraphModel(nn.Module):
    """
    GNN-based GNCA:
      - Input node feats: 12
         dynamic => 0..3
         static => 4..11
      - Input edge feats: 3
         dynamic => 0 => adjacency (0 or 1, we'll produce a logit)
                    1 => vlen
         static => 2 => W
      => output node feats => 12
      => output edge feats => 3
    """

    def __init__(
        self,
        node_in_dim=12,
        edge_in_dim=3,
        hidden_dim=64,
        num_layers=3,
    ):
        super().__init__()
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim
        self.node_in_dim = node_in_dim
        self.edge_in_dim = edge_in_dim

        # 1) initial node embedding
        self.node_emb = nn.Linear(node_in_dim, hidden_dim)

        # 2) stack of EdgeFeatureConv
        self.convs = nn.ModuleList()
        for _ in range(num_layers):
            self.convs.append(EdgeFeatureConv(hidden_dim, edge_in_dim, hidden_dim))

        # 3) final node head => 12 outputs
        self.node_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, node_in_dim)
        )

        # 4) final edge head => 3 outputs
        #    index0 => adjacency logit
        #    index1 => next vlen
        #    index2 => next W
        self.edge_head = nn.Sequential(
            nn.Linear(hidden_dim*2 + edge_in_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, edge_in_dim)
        )

    def forward(self, x, edge_index, edge_attr):
        """
        :param x: shape [N, 12]
        :param edge_index: [2, E]
        :param edge_attr: [E, 3]
        :return: (node_out, edge_out)
          node_out => shape [N, 12]
          edge_out => shape [E, 3]
        """
        # 1) embed
        h = self.node_emb(x)
        h = F.relu(h)

        # 2) pass through EdgeFeatureConv layers
        for conv in self.convs:
            h = conv(h, edge_index, edge_attr)

        # 3) node output => shape [N,12]
        node_out = self.node_head(h)

        # 4) edge output => shape [E,3]
        src, dst = edge_index
        h_src = h[src]
        h_dst = h[dst]
        edge_in = torch.cat([h_src, h_dst, edge_attr], dim=1)  # [E, hidden_dim*2 + 3]
        edge_out = self.edge_head(edge_in)

        return node_out, edge_out


In [32]:
import torch
import torch.nn.functional as F
import numpy as np

def gnca_train_fullgraph(
    model,
    dataset,
    device=torch.device("cpu"),
    lr=1e-3,
    epochs=20,
):
    """
    :param model: GNCAFullGraphModel
    :param dataset: list of (pyg_data, label_dict) for each time step
    :param device: CPU or CUDA
    :param lr: learning rate for Adam
    :param epochs: number of training epochs
    """
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # For node feats => dynamic are indices 0..3
    # For edge feats => index0 => adjacency => BCE, index1 => vlen => MSE, index2 => W => ignore
    for epoch in range(epochs):
        total_loss = 0.0
        total_samples = 0

        # Shuffle each epoch
        np.random.shuffle(dataset)

        for (pyg_data, label_dict) in dataset:
            pyg_data = pyg_data.to(device)

            node_target = torch.tensor(label_dict["node_feats"], dtype=torch.float, device=device)
            edge_target = torch.tensor(label_dict["edge_feats"], dtype=torch.float, device=device)

            optimizer.zero_grad()

            node_out, edge_out = model(
                pyg_data.x, pyg_data.edge_index, pyg_data.edge_attr
            )
            # node_out => [N,12]
            # edge_out => [E,3], where edge_out[:,0] => adjacency logit, edge_out[:,1] => vlen, edge_out[:,2] => W

            # 1) Node MSE => only indices 0..3
            node_mse = F.mse_loss(node_out[:,0:4], node_target[:,0:4])

            # 2) Edge adjacency => BCE on edge_out[:,0]
            # adjacency is either 0 or 1 in edge_target[:,0]
            edge_logits = edge_out[:,0]   # shape [E]
            adj_label   = edge_target[:,0]
            edge_bce = F.binary_cross_entropy_with_logits(edge_logits, adj_label)

            # 3) Edge vlen => MSE on edge_out[:,1]
            # target => edge_target[:,1], skip index2 => W
            edge_vlen_pred = edge_out[:,1]
            edge_vlen_targ = edge_target[:,1]
            edge_mse = F.mse_loss(edge_vlen_pred, edge_vlen_targ)

            loss = node_mse + edge_bce + edge_mse
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            total_samples += 1

        avg_loss = total_loss / total_samples
        print(f"Epoch {epoch+1}/{epochs}, Loss={avg_loss:.4f}")


In [33]:
import os
import pickle
import argparse
import torch
from torch_geometric.data import Data

def main_train_gnca():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data_dir", type=str, default="fullgraph_data",
        help="Folder with .pkl files of (pyg_data, label_dict).")
    parser.add_argument("--num_layers", type=int, default=3,
        help="Number of GNN layers => node sees radius=--num_layers.")
    parser.add_argument("--hidden_dim", type=int, default=64,
        help="Hidden dimension.")
    parser.add_argument("--epochs", type=int, default=20,
        help="Number of training epochs.")
    parser.add_argument("--lr", type=float, default=1e-3,
        help="Learning rate for Adam.")
    args = parser.parse_args()

    file_list = [f for f in os.listdir(args.data_dir) if f.endswith(".pkl")]
    dataset = []
    for fname in file_list:
        fpath = os.path.join(args.data_dir, fname)
        with open(fpath, "rb") as f:
            pyg_data, label_dict = pickle.load(f)
        dataset.append((pyg_data, label_dict))

    if not dataset:
        print("No data found in", args.data_dir)
        return

    sample_data, sample_label = dataset[0]
    print("Sample node features shape:", sample_data.x.shape)
    print("Sample edge features shape:", sample_data.edge_attr.shape)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    from torch import nn
    model = GNCAFullGraphModel(
        node_in_dim=12,
        edge_in_dim=3,
        hidden_dim=args.hidden_dim,
        num_layers=args.num_layers
    )

    gnca_train_fullgraph(
        model,
        dataset,
        device=device,
        lr=args.lr,
        epochs=args.epochs
    )

    print("Training complete! Now the model can predict node[12] + edge[3] => next step.")


if __name__ == "__main__":
    main_train_gnca()


usage: ipykernel_launcher.py [-h] [--data_dir DATA_DIR]
                             [--num_layers NUM_LAYERS]
                             [--hidden_dim HIDDEN_DIM] [--epochs EPOCHS]
                             [--lr LR]
ipykernel_launcher.py: error: unrecognized arguments: --f=/Users/sophia01px2019/Library/Jupyter/runtime/kernel-v3a9b10f90e167a1f47449c971c56f3e6bc02b6bea.json


SystemExit: 2

/Users/sophia01px2019/miniforge3/envs/new_env/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3558: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
